# NLU HOSTAGE — ai_1_nlu_v1

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [1]:
from pathlib import Path
DATASET_FILENAME = "v1_chat_dataset_100.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v1"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")


Dataset aktif: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v1\data\v1_chat_dataset_100.csv


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
# Kebijakan proyek: SVM/NB pada CPU, Transformer pada GPU CUDA.
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cuda"


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=4):
    return train_transformer(DATASET_PATH, MODEL_DIR, epochs=epochs, device=TRANSFORMER_DEVICE)

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

HOSTAGE_TEST_CASES = [
    ("accusing", "B kena Gag Order saat menjelaskan alibi, menurut gw itu pola Hitman."),
    ("defending", "Gw bukan Hitman, tuduhan itu gak punya bukti publik."),
    ("bluffing", "Gw Spy, semalam gw Guard Raka dan dia pasti aman."),
    ("probing", "Stalker, semalam lu Peek siapa dan hasilnya apa?"),
    ("deflecting", "Jangan fokus ke gw, cek D yang terus mengubah cerita tiap ditanya."),
    ("persuading", "Vote C aja, dia paling diuntungkan dari korban Hostage semalam."),
    ("claiming", "Klaim gw Civilian, gw gak punya skill malam."),
    ("neutral", "Fase malam bikin chat terkunci, kita tunggu pagi dulu."),
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),
train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().
Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.
Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.


In [3]:
# JALANKAN SEMUA MODEL: empat model CPU, lalu Transformer CUDA dan 10 chat uji.
RUN_TRANSFORMER = True
RUN_TUNING = False
TRANSFORMER_EPOCHS = 4
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)



MENJALANKAN: SVM baseline
SVM | train=648 | test=162 | kelas=8

--- Evaluasi SVM baseline (holdout test set) ---
Accuracy    : 0.6728
Macro F1    : 0.6654
Weighted F1 : 0.6655
              precision    recall  f1-score   support

    accusing       0.58      0.71      0.64        21
    bluffing       0.78      0.67      0.72        21
    claiming       0.62      0.75      0.68        20
   defending       0.67      0.70      0.68        20
  deflecting       0.52      0.60      0.56        20
     neutral       0.84      0.80      0.82        20
  persuading       0.67      0.30      0.41        20
     probing       0.77      0.85      0.81        20

    accuracy                           0.67       162
   macro avg       0.68      0.67      0.67       162
weighted avg       0.68      0.67      0.67       162

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v1\models\intent_classifier_svm.pkl

MENJALANKAN: Naive Bayes baseline


Naive Bayes | train=648 | test=162 | kelas=8

--- Evaluasi Naive Bayes baseline (holdout test set) ---
Accuracy    : 0.6420
Macro F1    : 0.6285
Weighted F1 : 0.6290
              precision    recall  f1-score   support

    accusing       0.56      0.90      0.69        21
    bluffing       0.55      0.76      0.64        21
    claiming       0.63      0.60      0.62        20
   defending       0.65      0.65      0.65        20
  deflecting       0.65      0.55      0.59        20
     neutral       1.00      0.50      0.67        20
  persuading       0.50      0.25      0.33        20
     probing       0.78      0.90      0.84        20

    accuracy                           0.64       162
   macro avg       0.67      0.64      0.63       162
weighted avg       0.66      0.64      0.63       162

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v1\models\intent_classifier_nb.pkl

MENJALANKAN: IndoBERT Transformer


C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Map:   0%|          | 0/648 [00:00<?, ? examples/s]

Map: 100%|██████████| 648/648 [00:00<00:00, 31831.22 examples/s]

...

expected=deflecting   | predicted=defending    | confidence= 20.21% | MISS
expected=persuading   | predicted=persuading   | confidence= 21.09% | OK
expected=claiming     | predicted=claiming     | confidence= 36.85% | OK
expected=neutral      | predicted=neutral      | confidence= 42.53% | OK
expected=accusing     | predicted=persuading   | confidence= 34.57% | MISS
expected=defending    | predicted=neutral      | confidence= 31.85% | MISS

--- 10 chat uji: IndoBERT Transformer ---


expected=accusing     | predicted=deflecting   | confidence= 66.06% | MISS
expected=defending    | predicted=defending    | confidence= 49.91% | OK
expected=bluffing     | predicted=deflecting   | confidence= 77.81% | MISS
expected=probing      | predicted=probing      | confidence= 93.72% | OK
expected=deflecting   | predicted=probing      | confidence= 29.88% | MISS
expected=persuading   | predicted=deflecting   | confidence= 67.58% | MISS
expected=claiming     | predicted=deflecting   | confidence= 92.27% | MISS
expected=neutral      | predicted=deflecting   | confidence= 77.91% | MISS
expected=accusing     | predicted=persuading   | confidence= 66.67% | MISS
expected=defending    | predicted=defending    | confidence= 36.07% | OK
RINGKASAN EVALUASI HOLDOUT:


,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,waktu_detik,status
0,IndoBERT Transformer,0.8457,0.8464,0.8453,33.8,berhasil
1,SVM baseline,0.6728,0.6654,0.6655,0.2,berhasil
2,Naive Bayes baseline,0.6420,0.6285,0.6290,0.1,berhasil


RINGKASAN 10 CHAT UJI:


,model,benar_dari_10,akurasi_10_chat
0,SVM baseline,7,0.7
1,Naive Bayes baseline,7,0.7
2,IndoBERT Transformer,3,0.3


## Laporan eksekusi notebook

Tuning SVM dan Naive Bayes dilewati untuk mempercepat run ini. Output training lengkap tersimpan pada cell tepat di atas.

| Model | Macro-F1 holdout | Uji 10 chat |
|---|---:|---:|
| SVM baseline | 0.6654 | 7/10 |
| Naive Bayes baseline | 0.6285 | 7/10 |
| IndoBERT Transformer (GPU) | 0.8464 | 3/10 |

Transformer terbaik pada holdout, tetapi SVM dan Naive Bayes lebih stabil pada 10 chat manual.